### Configuration, inputs & outputs

In [1]:
from pipeline_utils import make_case2_paths
import pandas as pd
import numpy as np
import warnings
from collections import Counter

warnings.filterwarnings("ignore")
np.random.seed(101)

# Configure paths
paths = make_case2_paths()

PROJECT_DIR = paths["project"]
RAW_DIR = paths["raw_pdxe"]
PROC_DIR = paths["proc"]
RESULTS_DIR = paths["results"]

print(f"Project directory: {PROJECT_DIR}")
print(f"Raw data directory: {RAW_DIR}")
print(f"Processed data directory: {PROC_DIR}")
print(f"Results directory: {RESULTS_DIR}")

# Input files
expert1_file = RAW_DIR / "expert_rating" / "pdx_results_expert1.json"
expert2_file = RAW_DIR / "expert_rating" / "pdx_results_expert2.json"
expert3_file = RAW_DIR / "expert_rating" / "pdx_results_expert3.json"

expert1 = pd.read_json(expert1_file)
expert2 = pd.read_json(expert2_file)
expert3 = pd.read_json(expert3_file)

print(f"Expert 1 ratings file: {expert1_file}")
print(f"Expert 2 ratings file: {expert2_file}")
print(f"Expert 3 ratings file: {expert3_file}")

# Output files
expert_rating_dir = PROC_DIR / "expert_rating"
expert_rating_dir.mkdir(parents=True, exist_ok=True)

expert_rating_table = expert_rating_dir / "expert_rating_summary.csv"
expert_rating_qc_table = expert_rating_dir / "expert_rating_qc.csv"

Project directory: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study
Raw data directory: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/rawdata/pdxe
Processed data directory: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/2_drugSensitivity
Results directory: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/results/2_drugSensitivity
Expert 1 ratings file: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/rawdata/pdxe/expert_rating/pdx_results_expert1.json
Expert 2 ratings file: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/rawdata/pdxe/expert_rating/pdx_results_expert2.json
Expert 3 ratings file: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/rawdata/pdxe/expert_rating/pdx_results_expert3.json


### Merge expert ratings across raters

In [2]:
df1 = pd.DataFrame(expert1["ratings"]).reset_index()
df1.columns = ["figure_id", "rating_expert1"]

df2 = pd.DataFrame(expert2["ratings"]).reset_index()
df2.columns = ["figure_id", "rating_expert2"]

df3 = pd.DataFrame(expert3["ratings"]).reset_index()
df3.columns = ["figure_id", "rating_expert3"]

merged_df = pd.merge(df1, df2, on="figure_id", how="outer")
merged_df = pd.merge(merged_df, df3, on="figure_id", how="outer")
merged_df = merged_df.sort_values("figure_id").reset_index(drop=True)

print(f"Expert 1 ratings: {df1.shape[0]}")
print(f"Expert 2 ratings: {df2.shape[0]}")
print(f"Expert 3 ratings: {df3.shape[0]}")
print(f"Merged figures (union of raters): {merged_df.shape[0]}")

Expert 1 ratings: 1300
Expert 2 ratings: 1297
Expert 3 ratings: 1289
Merged figures (union of raters): 1300


### Consolidate ratings by majority vote

In [3]:
def consolidate_ratings(row):
    """Return the majority-vote category, or the "/"-joined ratings on a 3-way tie."""
    ratings = [row["rating_expert1"], row["rating_expert2"], row["rating_expert3"]]
    valid_ratings = [str(r) for r in ratings if pd.notna(r)]

    if not valid_ratings:
        return "No Ratings"

    counts = Counter(valid_ratings)
    most_common = counts.most_common()  # [(value, count), ...]

    if most_common[0][1] >= 2:
        return most_common[0][0]
    else:
        return "/".join(valid_ratings)


merged_df["rating_merge"] = merged_df.apply(consolidate_ratings, axis=1)

print("Category counts after majority-vote consolidation:")
print(merged_df["rating_merge"].value_counts())

Category counts after majority-vote consolidation:
rating_merge
cat2              359
cat1              305
cat3              299
cat4              213
cat5               91
cat2/cat3/cat1      7
cat4/cat3/cat5      3
cat1/cat2           2
cat5/cat3/cat1      2
cat5/cat4/cat1      2
cat3/cat4/cat1      2
cat3/cat2/cat4      2
cat4/cat1/cat3      2
cat2/cat3/cat4      2
cat2/cat1           2
cat3/cat2/cat1      2
cat4/cat2/cat3      1
cat3/cat1           1
cat4/cat1           1
cat4/cat2/cat1      1
cat4/cat3/cat1      1
Name: count, dtype: int64


### Category counts by rater and consensus

In [4]:
categories = [f"cat{i}" for i in range(1, 6)]
is_majority = ~merged_df["rating_merge"].str.contains("/", na=False)

category_table = pd.DataFrame({
    "expert1": merged_df["rating_expert1"].value_counts(),
    "expert2": merged_df["rating_expert2"].value_counts(),
    "expert3": merged_df["rating_expert3"].value_counts(),
    "consensus": merged_df.loc[is_majority, "rating_merge"].value_counts(),
}).reindex(categories).fillna(0).astype(int).T

category_counts_table = expert_rating_dir / "expert_rating_category_counts.csv"
category_table.to_csv(category_counts_table)

print(f"Saved category counts by rater and consensus to {category_counts_table}")
category_table

Saved category counts by rater and consensus to /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/2_drugSensitivity/expert_rating/expert_rating_category_counts.csv


,cat1,cat2,cat3,cat4,cat5
expert1,233,435,282,250,100
expert2,255,423,362,233,24
expert3,387,289,300,174,139
consensus,305,359,299,213,91


### QC: rater agreement with consensus

In [5]:
total_figures = len(merged_df)
agree_expert1 = (merged_df["rating_expert1"] == merged_df["rating_merge"]).sum()
agree_expert2 = (merged_df["rating_expert2"] == merged_df["rating_merge"]).sum()
agree_expert3 = (merged_df["rating_expert3"] == merged_df["rating_merge"]).sum()

print(f"Total figures: {total_figures}")
print(f"expert1 matches consensus: {agree_expert1} ({agree_expert1 / total_figures:.1%})")
print(f"expert2 matches consensus: {agree_expert2} ({agree_expert2 / total_figures:.1%})")
print(f"expert3 matches consensus: {agree_expert3} ({agree_expert3 / total_figures:.1%})")

Total figures: 1300
expert1 matches consensus: 1120 (86.2%)
expert2 matches consensus: 1003 (77.2%)
expert3 matches consensus: 1098 (84.5%)


### QC: consensus consistency (full agreement vs. partial disagreement)

In [6]:
# A "/"-joined rating_merge means no two experts agreed (3-way tie)
is_different = merged_df["rating_merge"].str.contains("/", na=False)

num_different = is_different.sum()
num_consistent = (~is_different).sum()
pct_different = num_different / total_figures * 100
pct_consistent = num_consistent / total_figures * 100

print(f"Consistent among at least two experts: {num_consistent} ({pct_consistent:.2f}%)")
print(f"No majority among all three experts:   {num_different} ({pct_different:.2f}%)")

Consistent among at least two experts: 1267 (97.46%)
No majority among all three experts:   33 (2.54%)


### QC summary for expert-rating ground truth

In [7]:
expert_rating_qc = pd.Series({
    "total_figures": total_figures,
    "agree_expert1": agree_expert1,
    "agree_expert2": agree_expert2,
    "agree_expert3": agree_expert3,
    "consistent_majority": num_consistent,
    "no_majority": num_different,
})
expert_rating_qc.to_csv(expert_rating_qc_table, header=["value"])

print(f"Saved expert-rating QC summary to {expert_rating_qc_table}")
print(expert_rating_qc)

Saved expert-rating QC summary to /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/2_drugSensitivity/expert_rating/expert_rating_qc.csv
total_figures          1300
agree_expert1          1120
agree_expert2          1003
agree_expert3          1098
consistent_majority    1267
no_majority              33
dtype: int64


### Save consolidated rating (ground truth)

In [8]:
merged_df.to_csv(expert_rating_table, index=False)
print(f"Saved consolidated expert-rating ground truth to {expert_rating_table}")

Saved consolidated expert-rating ground truth to /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/2_drugSensitivity/expert_rating/expert_rating_summary.csv
